# **Prophet vs XGboost: Inflação central**

**Introdução do projeto**

Importação das bibliotecas pandas
criação de um dicionário com o código de cada um dos bancos de dados selecionados:
 * IPCA — Índice de Preços ao Consumidor Amplo (índice oficial de inflação do Brasil, calculado pelo IBGE)
 * SELIC — Sistema Especial de Liquidação e Custódia (taxa básica de juros da economia brasileira, definida pelo Banco Central a cada 45 dias pelo COPOM)
 * Câmbio — Taxa USD/BRL (quantos reais valem um dólar americano)
 * IGP-M — Índice Geral de Preços do Mercado (mede a variação de preços no atacado, na construção civil e no varejo, Tende a antecipar movimentos do IPCA, o que sobe no atacado hoje chega ao consumidor final depois.)

 após isso aplicamos uma estrutura de repetição para buscar o banco de dados por meio da API e seu código, a API do BCB rejeita requisições sem identificação de navegador. Nesse sentido, utilizamos um header para nos idenfticar e enviar um request para obtermos uma response(resposta), após isso lemos essa resposta em json e em seguida passamos para a forma de dataframe. Após isso, deixamos a coluna de data no formate datetime e caso as colunas de valores tenha algum erro, retorna um valor nulo no lugar ao invés do código simplesmente quebrar, renomeio cada coluna com seu nome e printo a quantidade de registros e o range da data


In [ ]:

import pandas as pd
import requests

codigos = {
    'ipca': 433,
    'selic': 4189,
    'cambio': 3698,
    'igpm': 189
}

dfs = {}
for nome, codigo in codigos.items():
    url = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados?formato=json'
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    df_temp = pd.DataFrame(response.json())
    df_temp['data'] = pd.to_datetime(df_temp['data'], format='%d/%m/%Y')
    df_temp['valor'] = pd.to_numeric(df_temp['valor'], errors='coerce')
    df_temp = df_temp.rename(columns={'valor': nome})
    dfs[nome] = df_temp

print("Séries carregadas:")
for nome, df in dfs.items():
    print(f"{nome}: {df.shape[0]} registros | {df['data'].min().date()} até {df['data'].max().date()}")

In [ ]:
for nome, df in dfs.items():
    print(f"\n{nome}:")
    print(df.tail())

Fazemos uma cópia necessária para garantir que o DataFrame original do dicionário dfs não seja alterado durante as transformações subsequentes. Nesse sentido, passamos esta cópia para dentro da variável df_master, após isso fazemos que para cada coluna cujos nomes estão na seleção, seja feito um merge tendo a coluna data como referência utilizando o método inner. Diante disso, temos um dataframe único com os valores cconectados pelas mesmas datas. 
Obs: The exchange rate series (code 3694) was identified as annual frequency and replaced by the monthly series (code 3698) prior to merging

In [ ]:
df_master = dfs['ipca'].copy()

for nome in ['selic', 'cambio', 'igpm']:
    df_master = df_master.merge(dfs[nome], on='data', how='inner')

print(df_master.shape)
print(df_master.head())
print(df_master.tail())

# **EDA**

O dataset foi filtrado a partir de janeiro de 2000. O período de hiperinflação anterior ao Plano Real (1994) produziu variações de magnitude completamente diferente do regime monetário atual. Incluir dados anteriores a 2000 introduziria padrões incompatíveis com a dinâmica inflacionária atual, comprometendo a generalização dos modelos. Ademais, o reset_index(drop=True) é aplicado após o filtro para que os índices sejam reiniciados a partir de zero. Sem isso, os índices do DataFrame filtrado manteriam a numeração original, por exemplo, começando em 132 em vez de 0, o que pode causar problemas em operações subsequentes que assumem índices sequenciais.

In [ ]:


df_master = df_master[df_master['data'] >= '2000-01-01'].reset_index(drop=True)
df_master.head()


In [ ]:
# Plotting

import plotly.express as px
def plotting(title, data, x, y, x_label, y_label ):
    fig = px.line(data, x=df_master[x], y=df_master[y], labels={x: x_label, y: y_label})

    fig.update_layout(template='simple_white', font=dict(size=18), title_text = title, width=650, title_x = 0.5, height = 400)

    fig.show()

plotting('Evolução do IPCA (2000-2025)', df_master, 'data', 'ipca', 'Date', 'Values')






In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
from matplotlib import pyplot as plt
ipca = df_master['ipca']
plt.figure(figsize=(10, 5))
plot_acf(ipca, lags=40, markersize=4)
plt.title('Acf')
plt.show()

In [ ]:
print(ipca.describe())
print(f"Valores negativos: {(ipca < 0).sum()}")
print(f"Valores zero: {(ipca == 0).sum()}")

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decompose = seasonal_decompose(ipca, model='additive', period=12)
decompose.plot()
plt.show()


# **Prophet**

In [ ]:
df_train = df_master[(df_master['data'] >= '2000-01-01') & (df_master['data'] <= '2025-12-01')].reset_index(drop=True)
df_test = df_master[(df_master['data'] >= '2026-01-01')].reset_index(drop=True)

print(f'Treino: {df_train.shape} | De {df_train['data'].min().date()} até {df_train['data'].max().date()}')
print(f'Teste: {df_test.shape} | De {df_test['data'].min().date()} até {df_test['data'].max().date()}')